# YOLOv8n Recall Tuning (Colab)

This notebook trains a recall-tuned `YOLOv8n`, runs a threshold sweep, exports the tuned checkpoint to TFLite, and saves the outputs to Google Drive.

Recommended workflow:
1. Upload the `ai-model/` folder to Google Drive
2. Set the runtime to GPU
3. Run all cells
4. Review the saved model, threshold sweep results, and exported TFLite artifact

Important: the notebook copies the project from Google Drive to local Colab storage before training. This is faster and avoids Google Drive multiprocessing/cache issues during dataset scanning.

In [ ]:
!pip install -q ultralytics torch torchvision opencv-python matplotlib pandas PyYAML

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

AI_MODEL_DIR = Path('/content/drive/MyDrive/ai-model')
LOCAL_WORK_DIR = Path('/content/ai-model-work')
LOCAL_RUNS_DIR = LOCAL_WORK_DIR / 'training_runs'
DRIVE_RUNS_DIR = AI_MODEL_DIR / 'training_runs'
RUN_NAME = 'yolov8n_recall_a'
USE_COS_LR = True
TRAIN_WORKERS = 2
RUN_HIGHER_IMGSZ_EXPERIMENT = False
HIGHER_IMGSZ_RUN_NAME = 'yolov8n_recall_b'
RUN_LIGHT_AUG_EXPERIMENT = False
LIGHT_AUG_RUN_NAME = 'yolov8n_recall_c'
THRESHOLD_OUTPUT_NAME = 'threshold_sweep_results.json'

assert AI_MODEL_DIR.exists(), f'Missing folder: {AI_MODEL_DIR}'
print('Drive project folder:', AI_MODEL_DIR)
print('Local work folder:', LOCAL_WORK_DIR)

In [ ]:
required_files = [
    AI_MODEL_DIR / 'data.yaml',
    AI_MODEL_DIR / 'train.py',
    AI_MODEL_DIR / 'sweep_thresholds.py',
    AI_MODEL_DIR / 'export.py',
    AI_MODEL_DIR / 'datasets' / 'pothole_combined',
]

for path in required_files:
    print(path, 'OK' if path.exists() else 'MISSING')
    assert path.exists(), f'Missing required path: {path}'

In [ ]:
import shutil

if LOCAL_WORK_DIR.exists():
    shutil.rmtree(LOCAL_WORK_DIR)

print('Copying project from Drive to local Colab storage...')
shutil.copytree(AI_MODEL_DIR, LOCAL_WORK_DIR)
print('Copy complete:', LOCAL_WORK_DIR)

In [ ]:
%cd /content/ai-model-work

train_cmd = [
    'python', 'train.py',
    '--name', RUN_NAME,
    '--project', str(LOCAL_RUNS_DIR),
    '--workers', str(TRAIN_WORKERS),
]

if USE_COS_LR:
    train_cmd.append('--cos-lr')

print('Running:', ' '.join(train_cmd))
!{' '.join(train_cmd)}

In [ ]:
baseline_model_path = LOCAL_RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
assert baseline_model_path.exists(), f'Missing model: {baseline_model_path}'

sweep_cmd = [
    'python', 'sweep_thresholds.py',
    '--model', str(baseline_model_path),
    '--output', str(LOCAL_WORK_DIR / THRESHOLD_OUTPUT_NAME),
]

print('Running:', ' '.join(sweep_cmd))
!{' '.join(sweep_cmd)}

In [ ]:
import json

with open(LOCAL_WORK_DIR / THRESHOLD_OUTPUT_NAME, 'r', encoding='utf-8') as handle:
    sweep_results = json.load(handle)

print('Best threshold setting:')
print(json.dumps(sweep_results['best'], indent=2))

In [ ]:
export_cmd = [
    'python', 'export.py',
    '--weights', str(baseline_model_path),
    '--output-dir', str(LOCAL_WORK_DIR / 'trained_model'),
    '--android-asset', '',
]

print('Running:', ' '.join(export_cmd))
!{' '.join(export_cmd)}

In [ ]:
if RUN_HIGHER_IMGSZ_EXPERIMENT:
    higher_imgsz_cmd = [
        'python', 'train.py',
        '--name', HIGHER_IMGSZ_RUN_NAME,
        '--project', str(LOCAL_RUNS_DIR),
        '--workers', str(TRAIN_WORKERS),
        '--imgsz', '704',
        '--batch', '12',
    ]
    if USE_COS_LR:
        higher_imgsz_cmd.append('--cos-lr')
    print('Running:', ' '.join(higher_imgsz_cmd))
    !{' '.join(higher_imgsz_cmd)}
else:
    print('Skipping higher image size experiment')

In [ ]:
if RUN_LIGHT_AUG_EXPERIMENT:
    light_aug_cmd = [
        'python', 'train.py',
        '--name', LIGHT_AUG_RUN_NAME,
        '--project', str(LOCAL_RUNS_DIR),
        '--workers', str(TRAIN_WORKERS),
        '--mosaic', '0.3',
        '--mixup', '0.0',
        '--degrees', '5',
        '--translate', '0.06',
        '--scale', '0.3',
    ]
    if USE_COS_LR:
        light_aug_cmd.append('--cos-lr')
    print('Running:', ' '.join(light_aug_cmd))
    !{' '.join(light_aug_cmd)}
else:
    print('Skipping lighter augmentation experiment')

In [ ]:
import shutil

DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

for run_dir in sorted(LOCAL_RUNS_DIR.glob('*')):
    destination = DRIVE_RUNS_DIR / run_dir.name
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(run_dir, destination)

shutil.copy2(LOCAL_WORK_DIR / THRESHOLD_OUTPUT_NAME, AI_MODEL_DIR / THRESHOLD_OUTPUT_NAME)
local_tflite = LOCAL_WORK_DIR / 'trained_model' / 'best_float16.tflite'
if local_tflite.exists():
    trained_model_dir = AI_MODEL_DIR / 'trained_model'
    trained_model_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local_tflite, trained_model_dir / 'best_float16.tflite')
print('Artifacts synced back to Google Drive')

In [ ]:
from pathlib import Path

training_runs_dir = AI_MODEL_DIR / 'training_runs'
print('Saved runs on Drive:')
for path in sorted(training_runs_dir.glob('*')):
    print('-', path)

print('Threshold sweep file:', AI_MODEL_DIR / THRESHOLD_OUTPUT_NAME)

## What to keep

After the run, keep these files from Google Drive:
- `training_runs/<run_name>/weights/best.pt`
- `training_runs/<run_name>/results.csv`
- `training_runs/<run_name>/results.png`
- `threshold_sweep_results.json`
- `trained_model/best_float16.tflite`

If recall improves without a large precision drop, that `best.pt` and its exported `best_float16.tflite` should become your new base model artifacts.